In [ ]:
# =========================
# Colab Tiny Transformer Trainer for Simple English Wikipedia
# Model: Llama-inspired, 4L, 256H, 4K vocab, RoPE, 256 window
# =========================

# --------- 0. Environment Setup ---------
# !pip install -U torch tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from datasets import load_dataset
# from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from tqdm import tqdm
import numpy as np
from datetime import datetime

# Distributed and parallel training
# import torch.distributed as dist
# import torch.multiprocessing as mp
# from torch.nn.parallel import DistributedDataParallel as DDP
# from torch.utils.data.distributed import DistributedSampler

## Colab - mount Google Drive
# from google.colab import files, drive
# drive.mount('/content/drive')

## Set models path
# Colab - Google Drive
# models_path = '/content/drive/My Drive/LanguageDynamics/models/'

# Remote (SSH) - Linux path
models_path = '/home/galk/LanguageDynamics/models/tinystories'

# Local - Windows path
# models_path = 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models'

In [ ]:
# --------- 1. Config ---------
### Language Modeling ###
# CONFIG = {
#     'mode': 'LM',
#     'n_layers': 6,
#     'n_heads': 4,
#     'embed_dim': 256,
#     'ffn_dim': 1024,
#     'vocab_size': 8000,
#     'context_window': 128,
#     'max_depth': 4,
#     'min_length': 4,
#     'max_length': 20,
#     'batch_size': 64,
#     'epochs': 500,
#     'lr': 1e-3,
#     'grad_clipping': True,
#     'save_every': 1,
#     'checkpoint_path': '/home/galk/LanguageDynamics/models/tiny_transformer_tinystories_weight_tied_epoch4_16_07_25.pt',
#     # 'checkpoint_path': None,
#     'device': 'cuda:1' if torch.cuda.is_available() else 'cpu'
# }

### Raw Language Model ###
# CONFIG = {
#     'mode': 'RLM',
#     'n_layers': 2,
#     'n_heads': 2,
#     'embed_dim': 32,
#     'ffn_dim': 256,
#     'vocab': ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>'],
#     'context_window': 32,
#     'max_depth': 4,
#     'min_length': 4,
#     'max_length': 50,
#     'batch_size': 128,
#     'epochs': 500,
#     'lr': 2e-4,
#     'save_every': 1,
#     'checkpoint_path': '/home/galk/LanguageDynamics/models/tiny_RLM_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_maxdepth_4_maxlen_50_date_200825_1742_trial_1/ckpt_epoch19.pt',
#     # 'checkpoint_path': None,
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu'
# }

# ## Autoencoder ###
# CONFIG = {
#     'mode': 'AE',
#     'n_layers': 2,
#     'n_heads': 2,
#     'embed_dim': 32,
#     'ffn_dim': 256,
#     'vocab': ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>'],
#     'context_window': 32,
#     'latent_dim': 32,
#     'n_latents': 4,
#     'max_depth': 4,
#     'min_length': 4,
#     'max_length': 50,
#     'batch_size': 512,
#     'epochs': 500,
#     'lr': 3e-4,
#     'save_every': 1,
#     # 'checkpoint_path': '/content/drive/MyDrive/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_100825_0456_trial_1/ckpt_epoch100.pt',
#     # 'checkpoint_path': 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_16_ffn_dim_128_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_080825_1626_trial_1/ckpt_epoch109.pt',
#     'checkpoint_path': None,
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu'
# }

## Koopman Autoencoder ###
CONFIG = {
    'mode': 'KAE',
    'n_layers': 6,
    'n_heads': 4,
    'embed_dim': 256,
    'ffn_dim': 1024,
    'vocab_size': 8002,
    'context_window': 128,
    'latent_dim': 128,
    'n_latents': 4,
    'n_diagonals': 10,  # Number of diagonals in the Koopman operator
    'max_depth': 4,
    'min_length': 4,
    'max_length': 20,
    'batch_size': 2,
    'epochs': 500,
    'lr': 1e-3,
    'reconstruction_coef': 10.0,
    'koopman_coef': 1.0,
    'regularization_coef': 0.001,
    'teacher_forcing': False,
    'grad_clipping': True,
    'save_every': 1,
    # 'checkpoint_path': '/content/drive/MyDrive/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_100825_0456_trial_1/ckpt_epoch100.pt',
    # 'checkpoint_path': 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_16_ffn_dim_128_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_080825_1626_trial_1/ckpt_epoch109.pt',
    # 'checkpoint_path': '/home/galk/LanguageDynamics/models/tiny_KAE_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_n_diagonals_5_maxdepth_4_maxlen_20_date_270825_1258_trial_1/ckpt_epoch36.pt',
    'checkpoint_path': None,
}

CONFIG['device'] = 'cuda:1' if torch.cuda.is_available() else 'cpu'
CONFIG['tokenizer_path'] = '/home/galk/LanguageDynamics/tokenizers/tokenizer_tinyllama_tinystories_vocab_8k_bytelevel_24_06_25.json'

current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
if CONFIG['mode'] == 'AE':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_latent_{CONFIG['latent_dim']}_n_latents_{CONFIG['n_latents']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"
if CONFIG['mode'] == 'KAE':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_latent_{CONFIG['latent_dim']}_n_latents_{CONFIG['n_latents']}_n_diagonals_{CONFIG['n_diagonals']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"
elif CONFIG['mode'] == 'LM' or CONFIG['mode'] == 'RLM':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"

## Overide model save prefix
# CONFIG['model_save_prefix'] = '/home/galk/LanguageDynamics/models/tiny_KAE_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_n_diagonals_5_maxdepth_4_maxlen_20_date_270825_1258'

# CONFIG['vocab_size'] = len(CONFIG['vocab'])

print(f"Using device: {CONFIG['device']}")
# print("Vocabulary:", CONFIG['vocab'])
print(f"{CONFIG['model_save_prefix']}")

In [ ]:
# --------- 2. Load and Prepare Data ---------
from datasets import load_dataset
print("Loading TinyStories dataset...")
ds = load_dataset("roneneldan/TinyStories", split="train")

# We'll concatenate all text for tokenizer training
# all_text = " ".join(ds['text'])
# print(f"Corpus length: {len(all_text)} characters.")

In [ ]:
from tokenizers import Tokenizer

# --------- 3. Train Custom Tokenizer ---------
if not os.path.exists(CONFIG['tokenizer_path']):
    print("Training custom tokenizer...")
    # Create the directory if it doesn't exist
    os.makedirs(CONFIG['tokenizer_dir'], exist_ok=True)

    # Define tokenizer
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=True)
    tokenizer.decoder = decoders.ByteLevel()

    # Define Trainer
    trainer = trainers.BpeTrainer(
        vocab_size=CONFIG['vocab_size'],
        special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    )
    def text_iterator(dataset):
      for item in dataset:
          yield item['text']

    # Train tokenizer
    # tokenizer.train_from_iterator([all_text[:50_000_000]], trainer)  # TRAIN ONLY ON 50M CHARACHTERS
    tokenizer.train_from_iterator(text_iterator(ds), trainer)

    # Save tokenizer
    tokenizer.save(f"{CONFIG['tokenizer_dir']}/tokenizer.json")
    print("Tokenizer trained and saved.")
else:
    print("Tokenizer already exists, loading.")
    tokenizer = Tokenizer.from_file(f"{CONFIG['tokenizer_path']}")

# Save for download
# !zip -r simplewiki_tokenizer.zip simplewiki_tokenizer
# files.download('simplewiki_tokenizer.zip')

pad_id = tokenizer.token_to_id("[PAD]")
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")
unk_id = tokenizer.token_to_id("[UNK]")
cls_id = CONFIG['vocab_size'] - 2
sos_id = CONFIG['vocab_size'] - 1

def encode(text):
    # Returns list of token ids
    return tokenizer.encode(text).ids



In [ ]:
# --------- 6. Transformer Model (Llama-inspired, RoPE) ---------
# RoPE implementation (from Llama paper and HF)
def apply_rope(x, base=10000.0, seq_dim=1):
    # x: [batch, seq, n_heads, head_dim]
    # RoPE mixes head_dim pairs
    batch, seq, n_heads, head_dim = x.size()
    half_dim = head_dim // 2
    pos = torch.arange(seq, dtype=torch.float32, device=x.device)
    idx = torch.arange(half_dim, dtype=torch.float32, device=x.device)
    freq = torch.exp(-math.log(base) * idx / half_dim)
    angles = pos[:, None] * freq[None, :]
    cos, sin = torch.cos(angles), torch.sin(angles)
    x1, x2 = x[..., :half_dim], x[..., half_dim:]
    x_rope = torch.cat([x1 * cos[None, :, None, :] - x2 * sin[None, :, None, :],
                        x1 * sin[None, :, None, :] + x2 * cos[None, :, None, :]], dim=-1)
    return x_rope

def generate_sinusoidal_embeddings(n_latents, embed_dim):
    pe = torch.zeros(n_latents, embed_dim)
    position = torch.arange(0, n_latents, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0) # [1, n_latents, embed_dim]

class RoPEMultiheadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, causal_mask=True, dropout=0.03):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.causal_mask = causal_mask

    def forward(self, x):
        # x: [batch, seq, embed_dim]
        B, S, E = x.shape

        # Create causal mask
        if self.causal_mask:
            causal_mask = torch.triu(torch.ones(S, S, device=x.device) * float('-inf'), diagonal=1)

        qkv = self.qkv_proj(x)                # [B, S, 3E]
        q, k, v = qkv.chunk(3, dim=-1)        # [B, S, E] each

        # reshape for multihead: [B, S, n_heads, head_dim]
        def split_heads(t):
            return t.view(B, S, self.n_heads, self.head_dim)
        q, k, v = map(split_heads, (q, k, v))

        # Apply RoPE to q and k
        q, k = apply_rope(q), apply_rope(k)

        # [B, n_heads, S, head_dim]
        q, k, v = [x.permute(0,2,1,3) for x in (q,k,v)]

        # Scaled dot-product attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        if self.causal_mask:
            attn_weights = attn_weights + causal_mask[None, None, :, :]  ## MASK ADDED BY CLAUDE
        attn_weights = self.dropout(attn_weights.softmax(dim=-1))  ## DROPOUT ADDED BY CLAUDE

        attn_output = torch.matmul(attn_weights, v)   # [B, n_heads, S, head_dim]
        attn_output = attn_output.permute(0,2,1,3).contiguous().view(B, S, E)
        return self.out_proj(attn_output)

class MultiheadCrossAttention(nn.Module):
    def __init__(self, embed_dim, latent_dim, n_latents, n_heads, dropout=0.03):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(latent_dim, embed_dim * n_latents, bias=False)
        self.v_proj = nn.Linear(latent_dim, embed_dim * n_latents, bias=False)
        # self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.n_latents = n_latents

        # Pre-compute and store sinusoidal positional encodings
        pe = generate_sinusoidal_embeddings(self.n_latents, embed_dim)
        self.register_buffer('positional_encodings', pe)

    def forward(self, x, z):
        # x: [batch, seq, embed_dim]
        # z: [batch, 1, latent_dim]
        B, S, E = x.shape

        q = self.q_proj(x)                # [B, S, E]
        k = self.k_proj(z)                # [B, 1, E * n_latents]
        v = self.v_proj(z)                # [B, 1, E * n_latents]

        k = k.view(B, self.n_latents, E)  # [B, n_latents, E]
        v = v.view(B, self.n_latents, E)  # [B, n_latents, E]
        # kv = kv.view(B, self.n_latents, E * 2)
        # k, v = kv.chunk(2, dim=-1)        # [B, n_latents, E] each
        # q, k, v = qkv.chunk(3, dim=-1)        # [B, S, E] each

        # Add positional encodings to the Keys and Values
        # The positional_encodings tensor is [1, n_latents, embed_dim] and will broadcast
        k = k + self.positional_encodings
        v = v + self.positional_encodings

        # reshape for multihead: [B, S, n_heads, head_dim]
        def split_heads(t):
            return t.view(B, t.size(1), self.n_heads, self.head_dim)
        q, k, v = map(split_heads, (q, k, v))

        # [B, n_heads, S, head_dim]
        q, k, v = [x.permute(0,2,1,3) for x in (q,k,v)]

        # Scaled dot-product attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn_weights = self.dropout(attn_weights.softmax(dim=-1))  ## DROPOUT ADDED BY CLAUDE

        attn_output = torch.matmul(attn_weights, v)   # [B, n_heads, S, head_dim]
        attn_output = attn_output.permute(0,2,1,3).contiguous().view(B, S, E)
        return self.out_proj(attn_output)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, dropout=0.03, causal_mask=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = RoPEMultiheadAttention(embed_dim, n_heads, causal_mask=causal_mask)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, latent_dim, n_latents, dropout=0.03):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = RoPEMultiheadAttention(embed_dim, n_heads, causal_mask=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.cross_attention = MultiheadCrossAttention(embed_dim, latent_dim, n_latents, n_heads)
        self.ln3 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, z):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.cross_attention(self.ln2(x), z))
        x = x + self.dropout(self.ffn(self.ln3(x)))
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, embed_dropout=0.03, normalize_before_projection=True):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim, causal_mask=False) for _ in range(n_layers)
        ])
        self.latent_projection = nn.Linear(embed_dim, latent_dim, bias=False)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.context_window = context_window
        self.cls_id = cls_id
        self.normalize_before_projection = normalize_before_projection


    def forward(self, x, return_internals=True):
        """
        Forward pass supporting token indices, probability distributions over tokens, or embedded vectors.

        Args:
            x (Tensor):
                - [B, T] if token indices (int)
                - [B, T, vocab_size] if probability distributions
                - [B, T, embed_dim] if pre-computed embeddings
            return_internals (bool): whether to return intermediate embeddings

        Returns:
            logits or (logits, initial_embeddings, final_embeddings)
        """
        B = x.size(0)
        T = x.size(1)
        E = self.embed.embedding_dim
        V = self.embed.num_embeddings

        if T > self.context_window:
            raise ValueError(f"Input sequence length {T} exceeds context window {self.context_window}")

        # Case 1: Token indices [B, T]
        if x.ndim == 2:
            # prepend [CLS] token
            x = torch.concat([torch.full((B, 1), self.cls_id, dtype=torch.long, device=x.device), x], dim=1)  # [B, T+1]
            x = self.embed(x)  # [B, T+1, E]

        # Case 2: Probability distributions over vocabulary [B, T, V]
        elif x.ndim == 3 and x.shape[2] == V:
            # Ensure it sums to 1 along the vocab dimension
            if not torch.allclose(x.sum(dim=2), torch.ones(B, T, device=x.device), atol=1e-4):
                raise ValueError("Input probabilities must sum to 1 along vocab dimension.")
            x = torch.matmul(x, self.embed.weight)  # [B, T, E]

        # Case 3: Precomputed embeddings [B, T, E]
        elif x.ndim == 3 and x.shape[2] == E:
            pass  # already in embedded space

        else:
            raise ValueError(f"Unrecognized input shape: {x.shape}")

        x = self.embed_dropout(x)
        initial_embeddings = x.clone()

        for layer in self.layers:
            x = layer(x)

        if self.normalize_before_projection:
            x = self.layer_norm(x)
        final_embeddings = x.clone()
        latent = self.latent_projection(x[:,0])  # project only the [CLS] token

        if return_internals:
            return latent, initial_embeddings, final_embeddings
        else:
            return latent

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents=8, embed_dropout=0.03):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(embed_dim, n_heads, ffn_dim, latent_dim, n_latents) for _ in range(n_layers)
        ])
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.ln_f = nn.LayerNorm(embed_dim)
        self.context_window = context_window
        self.sos_id = sos_id

    def forward(self, x, z, return_internals=True):
        # x is the decoding seed, shape [B, T]
        B, T = x.shape

        # prepend <SOS> token
        x = torch.concat([torch.full((B, 1), self.sos_id, dtype=torch.long, device=x.device), x], dim=1)  # [B, T+1]

        x = self.embed(x)
        x = self.embed_dropout(x)

        for layer in self.layers:
            x = layer(x, z)

        x = self.ln_f(x)
        final_embeddings = x.clone()
        logits = self.head(x)
        if return_internals:
            return logits, final_embeddings
        else:
            return logits

class TransformerAutoencoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, sos_id, n_latents=8, latent_dropout=0.03):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id)
        self.decoder = TransformerDecoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents)
        self.latent_dropout = nn.Dropout(latent_dropout)

        self.cls_id = cls_id
        self.sos_id = sos_id
        self.context_window = context_window

        # weight tying of embeddings and head
        self.decoder.embed.weight = self.encoder.embed.weight
        self.decoder.head.weight = self.encoder.embed.weight

    def forward(self, x, decoding_seed, return_internals=True):
        B, T = x.shape
        latent, initial_embeddings, final_embeddings = self.encoder(x, return_internals=return_internals)
        logits, final_embeddings = self.decoder(decoding_seed, self.latent_dropout(latent), return_internals=return_internals)

        if return_internals:
            return logits, latent, initial_embeddings, final_embeddings
        else:
            return logits, latent

class TinyLlamaTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_layers, n_heads, ffn_dim, context_window):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = None  # RoPE only
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight  # Optional head-embed weight tying
        self.context_window = context_window

    def forward(self, x, return_internals=False):
        """
        Forward pass supporting token indices, probability distributions over tokens, or embedded vectors.

        Args:
            x (Tensor):
                - [B, T] if token indices (int)
                - [B, T, vocab_size] if probability distributions
                - [B, T, embed_dim] if pre-computed embeddings
            return_internals (bool): whether to return intermediate embeddings

        Returns:
            logits or (logits, initial_embeddings, final_embeddings)
        """
        B = x.size(0)
        T = x.size(1)
        E = self.embed.embedding_dim
        V = self.embed.num_embeddings

        if T > self.context_window:
            raise ValueError(f"Input sequence length {T} exceeds context window {self.context_window}")

        # Case 1: Token indices [B, T]
        if x.ndim == 2:
            x = self.embed(x)  # [B, T, E]

        # Case 2: Probability distributions over vocabulary [B, T, V]
        elif x.ndim == 3 and x.shape[2] == V:
            # Ensure it sums to 1 along the vocab dimension
            if not torch.allclose(x.sum(dim=2), torch.ones(B, T, device=x.device), atol=1e-4):
                raise ValueError("Input probabilities must sum to 1 along vocab dimension.")
            x = torch.matmul(x, self.embed.weight)  # [B, T, E]

        # Case 3: Precomputed embeddings [B, T, E]
        elif x.ndim == 3 and x.shape[2] == E:
            pass  # already in embedded space

        else:
            raise ValueError(f"Unrecognized input shape: {x.shape}")

        initial_embeddings = x.clone()

        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)  #TODO: Replace final embeddings to be before the LayerNorm!
        final_embeddings = x.clone()
        logits = self.head(x)

        if return_internals:
            return logits, initial_embeddings, final_embeddings
        else:
            return logits

class TinyLlamaRawTransformer(nn.Module):
    def __init__(self, embed_dim, n_layers, n_heads, ffn_dim, context_window):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim) for _ in range(n_layers)
        ])
        self.embed_dim = embed_dim
        self.ln_f = nn.LayerNorm(embed_dim)
        self.context_window = context_window

    def forward(self, x, return_internals=False):
        """
        Forward pass supporting token indices, probability distributions over tokens, or embedded vectors.

        Args:
            x (Tensor):
                - [B, T] if token indices (int)
                - [B, T, vocab_size] if probability distributions
                - [B, T, embed_dim] if pre-computed embeddings
            return_internals (bool): whether to return intermediate embeddings

        Returns:
            logits or (logits, initial_embeddings, final_embeddings)
        """
        B = x.size(0)
        T = x.size(1)
        E = self.embed_dim

        if T > self.context_window:
            raise ValueError(f"Input sequence length {T} exceeds context window {self.context_window}")

        # Case 3: Precomputed embeddings [B, T, E]
        if x.ndim == 3 and x.shape[2] == E:
            pass  # already in embedded space

        else:
            raise ValueError(f"Unrecognized input shape: {x.shape}")

        initial_embeddings = x.clone()

        for layer in self.layers:
            x = layer(x)

        # final LayerNorm?
        # x = self.ln_f(x)
        # final_embeddings = x.clone()

        if return_internals:
            return x, initial_embeddings
        else:
            return x

In [ ]:
class KoopmanNet(nn.Module):
    def __init__(self, obsdim: int, n_diagonals: int = 2):
        super().__init__()
        self.obsdim = obsdim
        
        # Initialize diagonal parameters - linearly spaced from 1 to 0
        self.kMatrixDiag = nn.Parameter(torch.linspace(1, 0, self.obsdim))
        
        # Get upper triangular indices for off-diagonal elements
        rows, cols = torch.triu_indices(self.obsdim, self.obsdim, offset=1)
        
        # Only keep the first two off-diagonals
        mask = cols - rows <= n_diagonals
        self.register_buffer('row_idx', rows[mask])
        self.register_buffer('col_idx', cols[mask])
        
        # Initialize off-diagonal parameters
        self.kMatrixUT = nn.Parameter(0.1 * torch.rand(self.row_idx.size(0)))
        self.kMatrixLT = nn.Parameter(0.1 * torch.rand(self.row_idx.size(0)))

    def get_koopman_matrix(self) -> torch.Tensor:
        """Constructs the Koopman matrix from diagonal and off-diagonal parameters."""
        # Initialize zero matrix
        K = torch.zeros(self.obsdim, self.obsdim, device=self.kMatrixDiag.device)
        
        # Set diagonal elements
        K.diagonal().copy_(self.kMatrixDiag)
        
        # Set off-diagonal elements (upper triangle)
        K[self.row_idx, self.col_idx] = self.kMatrixUT
        # Set off-diagonal elements (lower triangle)
        K[self.col_idx, self.row_idx] = self.kMatrixLT
        
        return K

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Applies the Koopman operator to the observables."""
        K = self.get_koopman_matrix()
        z_predicted = torch.matmul(z, K.T)
        self.K = K  # Store the Koopman matrix for later use
        return z_predicted

    @property
    def koopmanMatrix(self, requires_grad: bool = True) -> torch.Tensor:
        """Returns the current Koopman matrix."""
        if requires_grad:
            return self.K
        else:
            return self.K.detach()
        
class TransformerKoopmanAutoencoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, sos_id, n_latents=8, latent_dropout=0.03, n_diagonals=5):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id)
        self.decoder = TransformerDecoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents)
        self.koopman = KoopmanNet(obsdim=latent_dim, n_diagonals=n_diagonals)
        self.latent_dim = latent_dim
        self.latent_dropout = nn.Dropout(latent_dropout)

        self.cls_id = cls_id
        self.sos_id = sos_id
        self.context_window = context_window

        # weight tying of embeddings and head
        self.decoder.embed.weight = self.encoder.embed.weight
        self.decoder.head.weight = self.encoder.embed.weight

    def forward(self, x, decoding_seed, return_internals=True):
        B, T = x.shape
        latent, initial_embeddings, final_embeddings = self.encoder(x, return_internals=return_internals)
        logits, final_embeddings = self.decoder(decoding_seed, self.latent_dropout(latent), return_internals=return_internals)

        if return_internals:
            return logits, latent, initial_embeddings, final_embeddings
        else:
            return logits, latent

In [ ]:
# Loading a saved model
ckpt_path = CONFIG['checkpoint_path']
# ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
checkpoint = torch.load(ckpt_path, map_location=torch.device(CONFIG['device']))

# Load Tokenizer
# tokenizer_path = "tokenizer_tinyllama_tinystories_vocab_8k_bytelevel_24_06_25.json"
tokenizer = Tokenizer.from_file(CONFIG['tokenizer_path'])

if 'mode' not in checkpoint['config']:
    checkpoint['config']['mode'] = "LM"  # Default to LM if not specified

if checkpoint['config']['mode'] == 'LM':
    model = TinyLlamaTransformer(
        vocab_size=checkpoint['config']['vocab_size'],
        embed_dim=checkpoint['config']['embed_dim'],
        n_layers=checkpoint['config']['n_layers'],
        n_heads=checkpoint['config']['n_heads'],
        ffn_dim=checkpoint['config']['ffn_dim'],
        context_window=checkpoint['config']['context_window']
    ).to(CONFIG['device'])

elif CONFIG['mode'] == 'RLM':
        model = TinyLlamaRawTransformer(
            embed_dim=CONFIG['embed_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window']
        ).to(CONFIG['device'])

elif checkpoint['config']['mode'] == 'AE':
    model = TransformerAutoencoder(
            vocab_size=checkpoint['config']['vocab_size'],
            embed_dim=checkpoint['config']['embed_dim'],
            latent_dim=checkpoint['config']['latent_dim'],
            n_layers=checkpoint['config']['n_layers'],
            n_heads=checkpoint['config']['n_heads'],
            ffn_dim=checkpoint['config']['ffn_dim'],
            context_window=checkpoint['config']['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=checkpoint['config']['n_latents']
        ).to(CONFIG['device'])
elif checkpoint['config']['mode'] == 'KAE':
    model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])


model_state = checkpoint['model_state_dict']
model.load_state_dict(model_state)

In [ ]:
# --------- 4. Encode Dataset ---------

# For speed, tokenize the entire corpus, then split into blocks
ds_tokenized = []
for t in tqdm(ds[:1_000_000]['text']):
    ds_tokenized.append([bos_id] + encode(t) + [eos_id])
# Remove rare long tails
# n_tokens = len(all_ids)
# n_blocks = n_tokens // CONFIG['context_window']

# data_blocks = np.array(all_ids[:n_blocks * CONFIG['context_window']]).reshape(n_blocks, CONFIG['context_window'])

# np.save(f"{CONFIG['tokenizer_dir']}/data_blocks.npy", data_blocks)

In [ ]:
import pickle
filename = '/home/galk/LanguageDynamics/data/tiny_stories_1M_tokenized.pkl'
with open(filename, 'wb') as file: # 'wb' for write binary mode
    pickle.dump(ds_tokenized, file)

In [ ]:
import torch
import torch.nn.functional as F
from typing import List, Optional, Union

@torch.no_grad()
def soft_generate_batch(
    model,
    prompt_tokens: Union[List[List[str]], torch.Tensor],
    num_steps: int,
    device,
    mode="soft",
    tokenizer=None,
    temperature=1
):
    """
    Batched soft autoregressive generation using expected embeddings instead of token sampling.
    Handles batch input for both token lists and embeddings.

    Args:
        model (TinyLlamaTransformer): The model instance.
        prompt_tokens (List[List[str]] or Tensor): 
            - List[List[str]]: batch of token sequences (strings).
            - Tensor: input embeddings of shape [B, T0, E].
        num_steps (int): Number of soft tokens to generate.
        device: device to use.
        mode: "hard", "soft", or "raw".
        tokenizer: tokenizer for encoding the tokens.
        temperature: Softmax temperature.

    Returns:
        input_embeddings: [B, T0 + num_steps, E]
        output_embeddings: [B, num_steps, E]
        distributions: [B, num_steps, V]
        generated_tokens: List[List[str]]  # argmax token IDs at each step for each batch
    """
    model.eval()

    # BOS/EOS ids
    if tokenizer is None:
        bos_id = token2id['<BOS>']
        eos_id = token2id['<EOS>']
    else:
        bos_id = tokenizer.token_to_id("[BOS]")
        eos_id = tokenizer.token_to_id("[EOS]")

    E = model.embed.embedding_dim
    V = model.embed.num_embeddings
    context_window = model.context_window

    # Handle prompt_tokens as batch of token lists
    if isinstance(prompt_tokens, list) and isinstance(prompt_tokens[0], list):
    #     batch_size = len(prompt_tokens)
    #     # Convert each sequence to ids
    #     if tokenizer is None:
    #         input_ids = [
    #             [bos_id] + [token2id[token] for token in seq]
    #             for seq in prompt_tokens
    #         ]
    #     else:
    #         input_ids = [
    #             [bos_id] + tokenizer.encode(''.join(seq)).ids
    #             for seq in prompt_tokens
    #         ]
    #     # Pad to same length
    #     max_len = max(len(ids) for ids in input_ids)
    #     input_ids = [
    #         ids + [pad_id] * (max_len - len(ids)) for ids in input_ids
    #     ]
        # input_ids = torch.tensor(input_ids, dtype=torch.long, device=device)  # [B, T0]

        # Assume that you are given a tokenized input with constant length
        input_ids = torch.tensor(prompt_tokens, dtype=torch.long, device=device)  # [B, T0]
        input_embeds = model.embed(input_ids)  # [B, T0, E]
        B, T0 = input_ids.shape
        generated_tokens = [seq.copy() for seq in prompt_tokens]
    elif isinstance(prompt_tokens, torch.Tensor):
        input_embeds = prompt_tokens.to(device)
        B, T0, E_check = input_embeds.shape
        assert E_check == E, f"Expected embedding dim {E}, got {E_check}"
        generated_tokens = [[] for _ in range(B)]
    else:
        raise ValueError("prompt_tokens must be List[List[str]] or Tensor [B, T0, E]")

    all_input_embeds = [input_embeds]
    output_embeds = []
    distributions = []

    for step in range(num_steps):
        current_input = torch.cat(all_input_embeds, dim=1)
        if current_input.size(1) > context_window:
            current_input = current_input[:, -context_window:, :]

        logits, _, final_embeds = model(current_input, return_internals=True)
        last_logits = logits[:, -1, :] / temperature  # [B, V]
        probs = F.softmax(last_logits, dim=-1)        # [B, V]
        last_out_embed = final_embeds[:, -1, :]       # [B, E]

        # Choose next input embedding based on mode
        if mode == "raw":
            next_input_embed = last_out_embed
        elif mode == "soft":
            next_input_embed = torch.matmul(probs, model.embed.weight)  # [B, E]
        elif mode == "hard":
            next_input_embed = model.embed(probs.argmax(dim=-1))  # [B, E]
        else:
            raise ValueError(f"Unknown mode: {mode}")

        # Record data
        output_embeds.append(last_out_embed)
        distributions.append(probs)
        # Update generated tokens for each batch
        for i in range(B):
            token_id = probs[i].argmax().item()
            if tokenizer is None:
                generated_tokens[i].append(id2token[token_id])
            else:
                # generated_tokens[i].append(tokenizer.decode([token_id], skip_special_tokens=False))
                generated_tokens[i].append(token_id)
        all_input_embeds.append(next_input_embed.unsqueeze(1))  # [B, 1, E]

    input_embeddings = torch.cat(all_input_embeds, dim=1)            # [B, T0 + num_steps, E]
    output_embeddings = torch.stack(output_embeds, dim=1)            # [B, num_steps, E]
    distributions = torch.stack(distributions, dim=1)                # [B, num_steps, V]

    return input_embeddings,output_embeddings, distributions, generated_tokens

In [ ]:
def stack_context_windows_tokens(trajectory_tokens, context_window: int, pad: bool = False):
    """
    Stack overlapping context windows from output_embeddings, with optional zero padding at the start.

    Args:
        output_embeddings (Tensor): [B, T, E] — output embedding sequences
        context_window (int): number of consecutive steps to stack
        pad (bool): If True, pad the first context_window - 1 steps with zeros so output has same length T.
                    If False, ignore the first context_window - 1 steps, resulting in output length T - context_window + 1.

    Returns:
        Tensor: [B, T', E * context_window] — stacked windowed embeddings
                where T' = T if pad=True, else T - context_window + 1
    """
    B, T = trajectory_tokens.shape

    if pad:
        # Pad with zeros at the beginning: [B, context_window - 1, E]
        padding = np.zeros(B, context_window - 1)
        padded_embeddings = np.cat([padding, trajectory_tokens], dim=1)  # shape: [B, T + context_window - 1, E]
        T_new = T
    else:
        if T < context_window:
            raise ValueError(f"Input sequence length T={T} is smaller than context_window={context_window}.")
        padded_embeddings = trajectory_tokens
        T_new = T - context_window + 1

    windows = []
    for t in range(T_new):
        # shape: [B, context_window]
        window = padded_embeddings[:, t:t + context_window]
        windows.append(window)

    # shape: [B, T', context_window]
    return np.stack(windows, axis=1)


In [ ]:
val_size = int(0.05 * len(ds_tokenized))  # 5% for validation
# indices = np.random.permutation(len(ds_tokenized))
# train_indices = indices[val_size:].tolist()
# val_indices = indices[:val_size].tolist()

train_seqs = ds_tokenized[val_size:]
val_seqs = ds_tokenized[:val_size]

In [ ]:
### Data Preperation for Autoencoder
def create_stacked_trajectories_array(initial_seqs, model, num_steps, context_window, device, generation_batch_size=16, tokenizer=None):
    """
    Generates trajectories from initial sequences using the model.
    
    Args:
        initial_seqs (list[lsit[int]]): List of initial sequences. For TinyStories, they are tokenized and with BOS prepended.
        model: The model to use for generation.
        num_steps (int): Number of steps to generate.
        context_window (int): Context window to use for the generation
        device: Device to run the model on.
        generation_batch_size (int): Size of each batch for generation.
    
    Returns:
        list: List of generated trajectories.
    """

    # filter only sequences that are long enough
    seqs_filtered = [seq[:context_window] for seq in initial_seqs if len(seq) >= context_window]
    n_batches = len(seqs_filtered) // generation_batch_size

    trajectories_list = []
    for batch in tqdm(range(n_batches)):
        x = seqs_filtered[(batch * generation_batch_size):((batch + 1) * generation_batch_size)]
        _, _, _, generated_tokens = soft_generate_batch(model, x, num_steps, device, mode="hard", tokenizer=tokenizer)
        trajectories_list.extend(generated_tokens)

    # encode the trajectories into a np.uint8 array
    # trajectories = np.zeros((len(trajectories_list), len(trajectories_list[0])), dtype=np.uint8)
    trajectories = np.array(trajectories_list, dtype=np.uint16)

    # for i, seq in enumerate(tqdm(trajectories_list)):
        # for j, token in enumerate(seq):
            # trajectories[i, j] = token2id[token]

    # prepend BOS token to each trajectory
    # trajectories = np.insert(trajectories, 0, bos_id, axis=1)  # Insert BOS token at the beginning of each trajectory

    # Stack context windows
    trajectories_stacked = stack_context_windows_tokens(trajectories, context_window=context_window)  # [trajectories, time, context_window]

    return trajectories_stacked


num_steps = 50  # Number of steps to generate, can be adjusted based on the model's context window
data_percentage = 0.1
train_stories = int(data_percentage * len(train_seqs))
val_stories = int(data_percentage * len(val_seqs))

train_trajectories_stacked = create_stacked_trajectories_array(
    initial_seqs=train_seqs[:train_stories],
    model=model,
    context_window=CONFIG['context_window'],
    num_steps=num_steps,
    device=CONFIG['device'],
    generation_batch_size=128,
    tokenizer=tokenizer
)

val_trajectories_stacked = create_stacked_trajectories_array(
    initial_seqs=val_seqs[:val_stories],
    model=model,
    context_window=CONFIG['context_window'],
    num_steps=num_steps,
    device=CONFIG['device'],
    generation_batch_size=128,
    tokenizer=tokenizer
)


In [ ]:
np.save('/home/galk/LanguageDynamics/data/tiny_stories_train_100000_stories_context_window_128_num_steps_50_tokenized_trajectories_stacked.npy', train_trajectories_stacked)
np.save('/home/galk/LanguageDynamics/data/tiny_stories_val_5000_stories_context_window_128_num_steps_50_tokenized_trajectories_stacked.npy', val_trajectories_stacked)

In [ ]:
# Save the trajectories to a HF dataset
from datasets import Dataset, DatasetInfo
train_trajectories_dataset = Dataset.from_dict(
    {"trajectory_ids": train_trajectories_stacked},
    #  dataset_info=DatasetInfo(
    #     description="Dyck2 Trajectories Dataset",
    #     features={
    #         "token2id": token2id,
    #         "id2token": id2token,
    #         "num_steps": num_steps,
    #         "config": CONFIG
    #     }
    # ),
    split="train"
)
train_trajectories_dataset.push_to_hub("Gal-Kinberg/LanguageDynamics-TinyStories", split="train", private=True)

val_trajectories_dataset = Dataset.from_dict(
    {"trajectory_ids": val_trajectories_stacked},
    # dataset_info=DatasetInfo(
    #     description="Dyck2 Trajectories Dataset",
    #     features={
    #         "token2id": token2id,
    #         "id2token": id2token,
    #         "num_steps": num_steps,
    #         "config": CONFIG
    #     }
    # ),
    split="validation"
)
val_trajectories_dataset.push_to_hub("Gal-Kinberg/LanguageDynamics-TinyStories", split="validation", private=True)



In [ ]:
from datasets import load_dataset
train_trajectories_dataset_loaded = load_dataset("Gal-Kinberg/LanguageDynamics-TinyStories", split="train").with_format("numpy")
val_trajectories_dataset_loaded = load_dataset("Gal-Kinberg/LanguageDynamics-TinyStories", split="validation").with_format("numpy")

In [ ]:
train_trajectories_stacked = np.load('/home/galk/LanguageDynamics/data/tiny_stories_train_100000_stories_context_window_128_num_steps_50_tokenized_trajectories_stacked.npy')
val_trajectories_stacked = np.load('/home/galk/LanguageDynamics/data/tiny_stories_val_5000_stories_context_window_128_num_steps_50_tokenized_trajectories_stacked.npy')

In [ ]:
# train_trajectories_np = train_trajectories_dataset_loaded['trajectory_ids']
# val_trajectories_np = val_trajectories_dataset_loaded['trajectory_ids']
train_trajectories_np = train_trajectories_stacked
val_trajectories_np = val_trajectories_stacked

In [ ]:
class TrajectoryDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # TODO: change to return a torch.Tensor?
        return self.dataset[idx]
    
train_trajectories_dataset_np = TrajectoryDataset(train_trajectories_np)
val_trajectories_dataset_np = TrajectoryDataset(val_trajectories_np)

In [ ]:
def collate_fn_np(batch):
    # 'batch' is a list of np arrays, of shape [T, C] of trajectory IDs
    trajs = np.array(batch)
    trajs = torch.tensor(trajs, dtype=torch.long)  # shape [B, T, C]
    return trajs

KAE_train_loader = DataLoader(
    train_trajectories_dataset_np, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    num_workers=16,
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True)

KAE_val_loader = DataLoader(
    val_trajectories_dataset_np, 
    batch_size=CONFIG['batch_size'], 
    shuffle=False, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    num_workers=16,
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True)


In [ ]:
# model = TransformerKoopmanAutoencoder(
#             vocab_size=CONFIG['vocab_size'],
#             embed_dim=CONFIG['embed_dim'],
#             latent_dim=CONFIG['latent_dim'],
#             n_layers=CONFIG['n_layers'],
#             n_heads=CONFIG['n_heads'],
#             ffn_dim=CONFIG['ffn_dim'],
#             context_window=CONFIG['context_window'],
#             cls_id=cls_id,
#             sos_id=sos_id,
#             n_latents=CONFIG['n_latents'],
#             n_diagonals=CONFIG['n_diagonals']
#         ).to(CONFIG['device'])
# model.train()
# x = train_trajectories_dataset_np[:1]
# x = torch.tensor(x, dtype=torch.long)
# # print(f"after batch loading: {torch.cuda.memory_allocated()}")
# # x is [B, T, C] tokens as tensors
# B, T, C = x.shape
# # reshape to [B*T, C] for model input
# x = x.view(B * T, C)
# x = x.to(CONFIG['device'])
# # print(f"After moving x: {torch.cuda.memory_allocated()}")
# y = x.clone() # for the masked reconstruction loss
# # print(f"After cloning y: {torch.cuda.memory_allocated()}")


# logits, latent, _, _ = model(x, decoding_seed=x[:,:-1]) # logits: [B*T, C, vocab_size], latent: [B*T, latent_dim]


In [ ]:
### Koopman Autoencoder Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_koopman_losses = []
        self.train_regularization_losses = []
        self.train_accuracies = []
        self.train_koopman_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_koopman_losses = []
        self.val_accuracies = []
        self.val_koopman_accuracies = []
        
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict['train_total_losses']
        self.train_reconstruction_losses = metrics_dict['train_reconstruction_losses']
        self.train_koopman_losses = metrics_dict['train_koopman_losses']
        self.train_regularization_losses = metrics_dict['train_regularization_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        try:
            self.train_koopman_accuracies = metrics_dict['train_koopman_accuracies']
        except KeyError:
            self.train_koopman_accuracies = []
        self.val_reconstruction_losses = metrics_dict['val_reconstruction_losses']
        self.val_koopman_losses = metrics_dict['val_koopman_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        try:
            self.val_koopman_accuracies = metrics_dict['val_koopman_accuracies']
        except KeyError:
            self.val_koopman_accuracies = []
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction and Koopman losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.train_koopman_losses, 
                    color=train_color, linestyle='--', label='Train Koopman')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].plot(self.epochs, self.val_koopman_losses, 
                    color=val_color, linestyle='--', label='Val Koopman')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction and Koopman Losses')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: Accuracies
        axes[0,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        try:
            axes[0,1].plot(self.epochs, self.train_koopman_accuracies, 
                        color=train_color, linestyle='--', label='Train Koopman')
        except :
            pass  # If train_koopman_accuracies is not available, skip this line
        axes[0,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        try:
            axes[0,1].plot(self.epochs, self.val_koopman_accuracies, 
                        color=val_color, linestyle='--', label='Val Koopman')
        except:
            pass  # If val_koopman_accuracies is not available, skip this line
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Accuracy')
        axes[0,1].set_title('Model Accuracies')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color)  # Green
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        axes[1,0].set_title('Total Training Loss')
        axes[1,0].grid(True)

        # Plot 4: Regularization loss
        axes[1,1].plot(self.epochs, self.train_regularization_losses, color=train_color)  # Orange
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Loss')
        axes[1,1].set_title('Training Regularization Loss')
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if CONFIG['mode'] == 'KAE':
        model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])

    else:
        raise ValueError(f"Unsupported mode: {CONFIG['mode']}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, CONFIG['model_save_prefix'] + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if CONFIG['checkpoint_path']:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                CONFIG['checkpoint_path'],
                model,
                CONFIG['device'],
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = CONFIG['lr']
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_koopman_loss = 0
        total_acc = 0
        total_koopman_acc = 0
        with torch.no_grad():
            for x in val_loader:
                # x is [B, T, C] tokens as tensors
                B, T, C = x.shape
                # reshape to [B*T, C] for model input
                x = x.view(B * T, C)
                x = x.to(device)
                y = x.clone() # for the masked reconstruction loss

                # Compute the reconstruction loss
                logits, latent, _, _ = model(x, decoding_seed=x[:,:-1]) # logits: [B*T, C, vocab_size], latent: [B*T, latent_dim]
                loss_reconstruction = criterion(logits.view(-1, CONFIG['vocab_size']), x.view(-1))

                if CONFIG['teacher_forcing']:
                    # advance all latents by one step using the koopman operator
                    latent_pred = model.koopman(latent)  # [B*T, latent_dim]
                
                else:
                    # initialize latent predictions
                    latent_pred = torch.zeros((B*T, model.latent_dim), device=device)
                    # get the first latent from each trajectory
                    curr_latent = latent[::T, :]  # [B, latent_dim]
                    # advance the first latent T timesteps using the koopman operator
                    for t in range(T):
                        curr_latent = model.koopman(curr_latent)  # [B, latent_dim]
                        # store the latent prediction for this timestep
                        latent_pred[t::T] = curr_latent
                
                # decode the latent predictions
                logits_koopman, _ = model.decoder(x[1:,:-1], model.latent_dropout(latent_pred[:-1]), return_internals=True)  # [B*T - 1, C, vocab_size]
                y[::T] = torch.full([C], fill_value=pad_id, dtype=torch.long)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)

                # y[::T].fill_(pad_id)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
                loss_koopman = criterion(logits_koopman.view(-1, CONFIG['vocab_size']), y[1:].view(-1))

                acc = calculate_accuracy(logits, x, pad_id)  # autoencoding reconstruction accuracy
                koopman_acc = calculate_accuracy(logits_koopman, y[1:], pad_id)  # koopman accuracy

                total_reconstruction_loss += loss_reconstruction.item()
                total_koopman_loss += loss_koopman.item()
                total_acc += acc
                total_koopman_acc += koopman_acc
        return total_reconstruction_loss / len(val_loader), total_koopman_loss / len(val_loader), total_acc / len(val_loader), total_koopman_acc / len(val_loader)

    # Training loop
    # print(f"Before Training: {torch.cuda.memory_allocated()}")
    for epoch in range(starting_epoch, CONFIG['epochs']):
        model.train()
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_koopman = 0.0
        epoch_loss_regularization = 0.0
        epoch_acc = 0.0
        epoch_koopman_acc = 0.0
        start = time.time()

        # Training phase
        # print(f"Before epoch start: {torch.cuda.memory_allocated()}")
        for batch, x in enumerate(tqdm(KAE_train_loader, desc=f"Epoch {epoch+1} Training")):
            # print(f"after batch loading: {torch.cuda.memory_allocated()}")
            # x is [B, T, C] tokens as tensors
            B, T, C = x.shape
            # reshape to [B*T, C] for model input
            x = x.view(B * T, C)
            x = x.to(CONFIG['device'])
            # print(f"After moving x: {torch.cuda.memory_allocated()}")
            # y = x.clone() # for the masked reconstruction loss
            # print(f"After cloning y: {torch.cuda.memory_allocated()}")
            
            
            logits, latent, _, _ = model(x, decoding_seed=x[:,:-1]) # logits: [B*T, C, vocab_size], latent: [B*T, latent_dim]
            # print(f"after forward: {torch.cuda.memory_allocated()}")
            loss_reconstruction = criterion(logits.view(-1, CONFIG['vocab_size']), x.view(-1))
            acc = calculate_accuracy(logits, x, pad_id)  # autoencoding reconstruction accuracy
            # print(f"after reconstruction loss: {torch.cuda.memory_allocated()}")

            if CONFIG['teacher_forcing']:
                # advance all latents by one step using the koopman operator
                latent_pred = model.koopman(latent)  # [B*T, latent_dim]
            
            else:
                # initialize latent predictions
                latent_pred = torch.zeros((B*T, model.latent_dim), device=CONFIG['device'])
                # get the first latent from each trajectory
                curr_latent = latent[::T, :]  # [B, latent_dim]
                # advance the first latent T timesteps using the koopman operator
                for t in range(T):
                    curr_latent = model.koopman(curr_latent)  # [B, latent_dim]
                    # store the latent prediction for this timestep
                    latent_pred[t::T] = curr_latent
            
            # print(f"After koopman prediction: {torch.cuda.memory_allocated()}")
            # decode the latent predictions
            logits_koopman, _ = model.decoder(x[1:,:-1], model.latent_dropout(latent_pred[:-1]), return_internals=True)  # [B*T - 1, C, vocab_size]
            # print(f"after Koopman decoding: {torch.cuda.memory_allocated()}")
            all_indices = torch.arange(x.shape[0], device=x.device)
            invalid_idx = (all_indices % T == 0)
            valid_idx = ~invalid_idx
            # saved_vals = x[::T].clone()
            # x[::T] = torch.full([C], fill_value=pad_id, dtype=torch.long)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
            # y[::T] = torch.full([C], fill_value=pad_id, dtype=torch.long)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
            

            # y[::T].fill_(pad_id)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
            # loss_koopman = criterion(logits_koopman.view(-1, CONFIG['vocab_size']), y[1:].view(-1))
            loss_koopman = criterion(logits_koopman[valid_idx[1:]].view(-1, CONFIG['vocab_size']), x[1:][valid_idx[1:]].view(-1))
            koopman_acc = calculate_accuracy(logits_koopman[valid_idx[1:]], x[1:][valid_idx[1:]], pad_id)  # koopman accuracy
            # print(f"After Koopman loss: {torch.cuda.memory_allocated()}")
            # x[::T] = saved_vals
                
            regularization_loss = torch.sum(torch.pow(model.koopman.koopmanMatrix, 2))

            loss = CONFIG['reconstruction_coef'] * loss_reconstruction + CONFIG['koopman_coef'] * loss_koopman + CONFIG['regularization_coef'] * regularization_loss

            optimizer.zero_grad()
            # print(f"After zero grad: {torch.cuda.memory_allocated()}")
            loss.backward()
            # print(f"After backwards: {torch.cuda.memory_allocated()}")

            # Optional: Gradient clipping
            if 'grad_clipping' in CONFIG and CONFIG['grad_clipping']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_loss_reconstruction += loss_reconstruction.item()
            epoch_loss_koopman += loss_koopman.item()
            epoch_loss_regularization += regularization_loss.item()
            epoch_acc += acc
            epoch_koopman_acc += koopman_acc

        # Validation phase
        val_reconstruction_loss, val_koopman_loss, val_acc, val_koopman_acc = validate(model, KAE_val_loader, criterion, x.device)

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(KAE_train_loader)
        train_acc = epoch_acc / len(KAE_train_loader)
        train_koopman_acc = epoch_koopman_acc / len(KAE_train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(KAE_train_loader)
        train_loss_koopman = epoch_loss_koopman / len(KAE_train_loader)
        train_loss_regularization = epoch_loss_regularization / len(KAE_train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_koopman_losses.append(train_loss_koopman)
        metrics.train_regularization_losses.append(train_loss_regularization)
        metrics.train_accuracies.append(train_acc)
        metrics.train_koopman_accuracies.append(train_koopman_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_koopman_losses.append(val_koopman_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.val_koopman_accuracies.append(val_koopman_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Koopman Acc: {train_koopman_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train Koopman Loss: {train_loss_koopman:.4f} | Train Regularization Loss: {train_loss_regularization:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val Koopman Loss: {val_koopman_loss:.4f} | Val Acc: {val_acc:.4f} | Val Koopman Acc: {val_koopman_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % CONFIG['save_every'] == 0 or (epoch+1) == CONFIG['epochs']:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': CONFIG,
                'epoch': epoch+1,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_koopman_losses': metrics.train_koopman_losses,
                    'train_regularization_losses': metrics.train_regularization_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'train_koopman_accuracies': metrics.train_koopman_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_koopman_losses': metrics.val_koopman_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'val_koopman_accuracies': metrics.val_koopman_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
!printenv "PYTORCH_CUDA_ALLOC_CONF"ds
# !export PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"

In [ ]:
!nvidia-smi -l 1